## Sandbox Agents


In [20]:
from pathlib import Path  # noqa: I001

from agents import Runner
from agents.run import RunConfig

from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig, SandboxPathGrant
from agents.sandbox.entries import LocalDir
from agents.sandbox.capabilities import Capabilities
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient

from rich.console import Console

In [5]:
OPENAI_MODEL_NAME = "gpt-5.4-nano"

console = Console()

In [35]:
CODE_DIR = Path("sandbox_agents/code").resolve()
assert CODE_DIR.is_dir(), f"Directory not found: {CODE_DIR}"

OUTPUT_DIR = Path("sandbox_agents/output").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [36]:
instructions = f"""
You are a software engineer that fixes bugs.
Review files in the sandbox code directory.

Write the fixed version of the file to this host output directory:
{OUTPUT_DIR}

Use full file paths when writing output.
Respond with a summary of what you did.
"""

In [37]:
manifest = Manifest(
    entries={"code": LocalDir(src=CODE_DIR)},
    extra_path_grants=[SandboxPathGrant(path=str(OUTPUT_DIR))],
)

capabilities = Capabilities.default()

In [38]:
capabilities

[Filesystem(type='filesystem', session=None, run_as=None, configure_tools=None),
 Shell(type='shell', session=None, run_as=None, configure_tools=None),
 Compaction(type='compaction', session=None, run_as=None, policy=None)]

In [39]:
run_config = RunConfig(
    sandbox=SandboxRunConfig(client=UnixLocalSandboxClient()),
    workflow_name="Sandbox coding example",
)

In [40]:
agent = SandboxAgent(
    name="Engineer",
    instructions=instructions,
    model="gpt-5.4-mini",
    default_manifest=manifest,
    capabilities=capabilities,
)

In [41]:
result = await Runner.run(agent, "Fix the bug in the code", run_config=run_config)
print(result.final_output)

Fixed `orders.py` by replacing the shared-list `dict.fromkeys` pattern with per-customer lists using `setdefault`.

Wrote the corrected file to:
`/Users/AnirbanGuha/Library/CloudStorage/GoogleDrive-guhaa1@gmail.com/My Drive/GitHub/The Complete Agent and MCP Course/Exercises/sandbox_agents/output/orders.py`
